In [1]:
%load_ext jupyter_black

In [2]:
import pickle
import vedo
import ovrlpy
import numpy as np

In [3]:
with open("data/analysis.pickle", "rb") as file:
    dataset = pickle.load(file)

In [4]:
doublets = dataset.detect_doublets(min_signal=3, integrity_sigma=2)
x, y = doublets["x", "y"].row(0)

In [5]:
window_transcripts = dataset.subset_transcripts(x, y, window_size=60)
xyz = window_transcripts[["x", "y", "z"]].to_pandas().values
_, rgb = dataset.transform_transcripts(window_transcripts)

rgb_uint8 = (rgb * 255).astype(np.uint8)
hex_colors = []
for i in range(len(rgb_uint8)):
    r, g, b = rgb_uint8[i]
    hex_colors.append((int(r) << 16) | (int(g) << 8) | int(b))

In [6]:
# opens the correct plot, but when closing it it freezes the view
# vedo.settings.default_backend = "ipyvtklink"
# # vedo.settings.default_backend = "k3d"

# list_of_points = xyz.tolist()
# vedo_points = vedo.Points(list_of_points, r=8)

# # vedo_points.pointdata["cell_id"] = window_transcripts["cell_id"].to_numpy()
# # vedo_points.cmap("tab20", "cell_id")

# # vedo_points.pointdata["gene"] = window_transcripts["gene"].to_pandas().cat.codes
# # vedo_points.cmap("tab20", "gene")
# # vedo_points.pointcolors = (np.array(rgb) * 255).astype(np.uint8)

# # Assign colors to points
# # vedo_points.pointdata["colors"] = np.array(hex_colors, dtype=np.uint32)
# # vedo_points.pointcolors = np.array(hex_colors, dtype=np.uint32).reshape(-1, 1)

# vedo.show(vedo_points)

In [7]:
import k3d

plot = k3d.plot(name="RGB colored points")

point_cloud = k3d.points(
    positions=xyz.astype(np.float32),
    colors=hex_colors,
    point_size=1,
    shader="flat",
)

plot += point_cloud
plot

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

The Xenium dataset has 50M transcripts. The performance of `k3d` is poor with such dimensionality.